In [1]:
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import CreateTable, CreateTableColumn, CreateTableConstraints, CreateForeignKey
import pandas as pd
from pandas.core.interchange.dataframe_protocol import DataFrame
from dotenv import load_dotenv
import os 
from dbrepo.api.dto import CreateView
from dbrepo.api.dto import CreateView, Subset, SubsetColumn, Join
from dbrepo.api.dto import JoinType

load_dotenv()
password = os.getenv("DBREPO_PASS")
username = os.getenv("DBREPO_USER")
client = RestClient("https://test.dbrepo.tuwien.ac.at/", username=username, password=password)

containers = client.get_containers()
print(containers)


[ContainerBrief(id='6cfb3b8e-1792-4e46-871a-f3d103527203', name='mariadb-galera:11.3.2', image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False), internal_name='mariadb_11_3_2', running=None, hash=None)]


In [3]:
DB_ID = "cf27a11d-58e5-4693-856c-e8f3527e3394"

In [4]:
df = client.get_database(DB_ID)
db_tables = client.get_tables(DB_ID)

In [6]:
tab_name_to_id = dict()
for i in db_tables:
    tab_name_to_id[i.name] = i.id

In [7]:
tab_name_to_id

{'wastewater_data': 'aa6cbf8c-f35e-4411-81ae-20f1a1acf682',
 'gdp_data': '0016c161-ead4-49ce-b8ea-a48c2797eeab',
 'city_map': '6c1a3235-df5c-4bc8-bbd0-b7dce088fcfe'}

In [12]:
tab_id_and_col_to_col_id = dict()

for tab_id in tab_name_to_id.values():
    table = client.get_table(DB_ID, tab_id)
    for col in table.columns:
        tab_id_and_col_to_col_id[(tab_id, col.name)] = col.id

In [24]:
table = client.get_table(DB_ID, tab_name_to_id["city_map"])
for col in table.columns:
    print(col)

id='70571097-af37-4cd3-9916-270cb3d4548b' name='nuts_code' database_id='cf27a11d-58e5-4693-856c-e8f3527e3394' table_id='6c1a3235-df5c-4bc8-bbd0-b7dce088fcfe' ord=0 internal_name='nuts_code' is_null_allowed=False type=<ColumnType.VARCHAR: 'varchar'> alias=None description='5-character NUTS-3 administrative code (e.g., AT221): https://ec.europa.eu/eurostat/web/nuts' size=5 d=None mean=None median=None concept=None unit=None enums=[] sets=[] index_length=None length=None data_length=None max_data_length=None num_rows=None val_min=None val_max=None std_dev=None
id='e0ddcf5f-c7b7-438e-be85-6c27c94d4a69' name='city_name' database_id='cf27a11d-58e5-4693-856c-e8f3527e3394' table_id='6c1a3235-df5c-4bc8-bbd0-b7dce088fcfe' ord=1 internal_name='city_name' is_null_allowed=False type=<ColumnType.VARCHAR: 'varchar'> alias=None description='The name of the city from EUDA/SCODA data (e.g., Graz)' size=100 d=None mean=None median=None concept=None unit=None enums=[] sets=[] index_length=None length=None

# Create View 1: Summary of drugs in water

In [14]:
wtable_id = tab_name_to_id["wastewater_data"]
df_city_summary_view = CreateView(
    name="vw_city_year_drug_summary",
    description="City-level aggregated wastewater indicators per year",
    is_public=True,
    is_schema_public=True,
    query=Subset(
        datasource_ids=[wtable_id],  # wastewater_data table ID

        columns=[
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "city_name")],
                alias="city_name"
            ),
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "ref_year")],
                alias="ref_year"
            ),

            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "daily_mean")],
                aggregation="avg",
                alias="avg_daily_mean"
            ),
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "daily_mean")],
                aggregation="max",
                alias="max_daily_mean"
            ),

            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "metabolite_name")],
                aggregation="count_distinct",
                alias="metabolite_count"
            )
        ],

        joins=None,
        filters=None,
        orders=None
    )
)


In [16]:
response = client._wrapper(
    method="post",
    url=f"/api/v1/database/{DB_ID}/view",
    payload=df_city_summary_view
)

print(response.status_code)
print(response.text)

201
{"id":"bc2d1e1a-8ca9-487e-b44b-0bfbc223f0eb","name":"vw_city_year_drug_summary","query":"select `dast_g20_wastewater_epidemiology_1kph`.`wastewater_data`.`metabolite_name` as `metabolite_count`, `dast_g20_wastewater_epidemiology_1kph`.`wastewater_data`.`city_name` as `city_name`, `dast_g20_wastewater_epidemiology_1kph`.`wastewater_data`.`daily_mean` as `max_daily_mean`, `dast_g20_wastewater_epidemiology_1kph`.`wastewater_data`.`ref_year` as `ref_year` from `wastewater_data`","database_id":"cf27a11d-58e5-4693-856c-e8f3527e3394","internal_name":"vw_city_year_drug_summary","is_public":true,"is_schema_public":true,"initial_view":false,"query_hash":"32de414d6d0557b88489b84ef618c8cadb3efa2992d40ae5f93e7b45c05a739f","owned_by":"data_stewardship_group20"}


# Create View 2: Join all

In [18]:
table = client.get_table(DB_ID, tab_name_to_id["city_map"])
print(table)  # or the UUID of the table
for col in table.columns:
    print(col.id, col.name)

id='6c1a3235-df5c-4bc8-bbd0-b7dce088fcfe' database_id='cf27a11d-58e5-4693-856c-e8f3527e3394' name='city_map' owner=UserBrief(username='data_stewardship_group20', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None) columns=[Column(id='70571097-af37-4cd3-9916-270cb3d4548b', name='nuts_code', database_id='cf27a11d-58e5-4693-856c-e8f3527e3394', table_id='6c1a3235-df5c-4bc8-bbd0-b7dce088fcfe', ord=0, internal_name='nuts_code', is_null_allowed=False, type=<ColumnType.VARCHAR: 'varchar'>, alias=None, description='5-character NUTS-3 administrative code (e.g., AT221): https://ec.europa.eu/eurostat/web/nuts', size=5, d=None, mean=None, median=None, concept=None, unit=None, enums=[], sets=[], index_length=None, length=None, data_length=None, max_data_length=None, num_rows=None, val_min=None, val_max=None, std_dev=None), Column(id='e0ddcf5f-c7b7-438e-be85-6c27c94d4a69', name='city_name', database_id='cf27a11d-58e5-4693-856c-e8f3527e3394', table_id='6c1a3235-df5c

In [19]:
table = client.get_table(DB_ID, tab_name_to_id["gdp_data"])
print(table) 
for col in table.columns:
    print(col.id, col.name)

id='0016c161-ead4-49ce-b8ea-a48c2797eeab' database_id='cf27a11d-58e5-4693-856c-e8f3527e3394' name='gdp_data' owner=UserBrief(username='data_stewardship_group20', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None) columns=[Column(id='85b5a0d7-845c-4a66-9ffb-20d2147f6d76', name='nuts_code', database_id='cf27a11d-58e5-4693-856c-e8f3527e3394', table_id='0016c161-ead4-49ce-b8ea-a48c2797eeab', ord=0, internal_name='nuts_code', is_null_allowed=False, type=<ColumnType.VARCHAR: 'varchar'>, alias=None, description='5-character NUTS-3 administrative code (e.g., AT221): https://ec.europa.eu/eurostat/web/nuts', size=5, d=None, mean=None, median=None, concept=None, unit=None, enums=[], sets=[], index_length=None, length=None, data_length=None, max_data_length=None, num_rows=None, val_min=None, val_max=None, std_dev=None), Column(id='9741a260-9bd8-488a-b6cd-bca545086614', name='ref_year', database_id='cf27a11d-58e5-4693-856c-e8f3527e3394', table_id='0016c161-ead4-

In [30]:
df_ml_view = CreateView(
    name="drug_gdp_features_view",
    description="ML-ready dataset joining wastewater measurements with GDP via city-NUTS mapping",
    is_public=True,
    is_schema_public=True,
    query=Subset(
        datasource_ids=[
            wtable_id  # wastewater_data
        ],

        columns=[
            # wastewater_data
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "city_name")],  # city_name
                alias="city_name"
            ),
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "ref_year")],  # ref_year
                alias="ref_year"
            ),
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "metabolite_name")],  # metabolite_name
                alias="metabolite_name"
            ),
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "daily_mean")],  # daily_mean
                alias="daily_mean"
            ),

            # city_map.nuts_code
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(tab_name_to_id["city_map"], "nuts_code")],  # nuts_code in city_map
                alias="nuts_code",
                #join_alias="m"
            ),

            # gdp_data.gdp_per_cap
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(tab_name_to_id["gdp_data"], "gdp_per_cap")],  # gdp_per_cap
                alias="gdp_per_cap",
                #join_alias="g"
            )
        ],

        joins=[
            # wastewater_data → city_map
            Join(
                type=JoinType.INNER,
                datasource_id=tab_name_to_id["city_map"],
                #alias="m",
                conditionals=[
                    {
                        "column_id": tab_id_and_col_to_col_id[(wtable_id, "city_name")],        # wastewater.city_name
                        "foreign_column_id": tab_id_and_col_to_col_id[(tab_name_to_id["city_map"], "city_name")] # city_map.city_name
                    }
                ]
            ),

            # city_map → gdp_data
            Join(
                type=JoinType.INNER,
                datasource_id=tab_name_to_id["gdp_data"],
                #alias="g",
                conditionals=[
                    {
                        "foreign_column_id": tab_id_and_col_to_col_id[(tab_name_to_id["city_map"], "nuts_code")],        # city_map.nuts_code
                        "column_id": tab_id_and_col_to_col_id[(tab_name_to_id["gdp_data"], "nuts_code")] # gdp.nuts_code
                    }
                ]
            )
                    ],

        filters=None,
        orders=None
    )
)



In [31]:
response = client._wrapper(
    method="post",
    url=f"/api/v1/database/{DB_ID}/view",
    payload=df_ml_view
)

print(response.status_code)
print(response.text)

201
{"id":"f9f564db-6fcb-470d-be2e-0b69a71fe588","name":"drug_gdp_features_view","query":"select `dast_g20_wastewater_epidemiology_1kph`.`city_map`.`nuts_code` as `nuts_code`, `dast_g20_wastewater_epidemiology_1kph`.`wastewater_data`.`metabolite_name` as `metabolite_name`, `dast_g20_wastewater_epidemiology_1kph`.`wastewater_data`.`city_name` as `city_name`, `dast_g20_wastewater_epidemiology_1kph`.`gdp_data`.`gdp_per_cap` as `gdp_per_cap`, `dast_g20_wastewater_epidemiology_1kph`.`wastewater_data`.`daily_mean` as `daily_mean`, `dast_g20_wastewater_epidemiology_1kph`.`wastewater_data`.`ref_year` as `ref_year` from `wastewater_data` join `city_map` on `dast_g20_wastewater_epidemiology_1kph`.`wastewater_data`.`city_name` = `dast_g20_wastewater_epidemiology_1kph`.`city_map`.`city_name` join `gdp_data` on `dast_g20_wastewater_epidemiology_1kph`.`gdp_data`.`nuts_code` = `dast_g20_wastewater_epidemiology_1kph`.`city_map`.`nuts_code`","database_id":"cf27a11d-58e5-4693-856c-e8f3527e3394","interna